In [3]:
import csv
import json
import math
from typing import Dict, List, Tuple, Any

# =========================
# Config
# =========================
ROADS_GEOJSON = "roads.geojson"
FULL_GEOJSON = "full.geojson"

NODES_CSV = "nodes.csv"
EDGES_CSV = "edges.csv"
POIS_CSV = "pois.csv"

DEFAULT_SPEED_KPH = {
    "motorway": 100.0,
    "trunk": 80.0,
    "primary": 60.0,
    "secondary": 50.0,
    "tertiary": 40.0,
    "residential": 25.0,
    "service": 15.0,
    "living_street": 10.0,
    "unclassified": 25.0,
}

ROADLIKE_HIGHWAYS = {
    "motorway",
    "trunk",
    "primary",
    "secondary",
    "tertiary",
    "residential",
    "unclassified",
    "service",
    "living_street",
}

NOISY_HIGHWAYS = {
    "crossing",
    "stop",
    "traffic_signals",
    "turning_circle",
    "turning_loop",
    "give_way",
    "motorway_junction",
    "speed_camera",
    "toll_gantry",
    "mini_roundabout",
    "bus_stop",
    "speed_display",
    "proposed",
    "stop;crossing",
    "sidewalk;crossing",
    "traffic_signals;crossing",
    "proposed;traffic_signals",
    "turning_circle;crossing",
    "trailhead",
    "passing_place",
}

INTERESTING_POI_KEYS = {
    "amenity",
    "tourism",
    "railway",
    "public_transport",
    "aeroway",
    "shop",
    "leisure",
    "office",
    "building",
    "addr:street",
    "addr:housenumber",
    "addr:city",
    "name",
}


# =========================
# Utility
# =========================
def load_geojson(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_get(props: Dict[str, Any], key: str, default: Any = "") -> Any:
    return props.get(key, default)


def coord_key(lon: float, lat: float) -> str:
    return f"{lon:.7f},{lat:.7f}"


def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2.0) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2.0) ** 2
    )
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return r * c


def parse_maxspeed_to_kph(maxspeed_value: Any) -> float:
    if maxspeed_value is None:
        return -1.0

    s = str(maxspeed_value).strip().lower()
    if not s:
        return -1.0

    if s.replace(".", "", 1).isdigit():
        return float(s)

    if "mph" in s:
        num = "".join(ch for ch in s if ch.isdigit() or ch == ".")
        if num:
            return float(num) * 1.60934
        return -1.0

    if "km/h" in s or "kph" in s:
        num = "".join(ch for ch in s if ch.isdigit() or ch == ".")
        if num:
            return float(num)
        return -1.0

    return -1.0


def resolve_speed_kph(highway: str, maxspeed: Any) -> Tuple[float, str]:
    parsed = parse_maxspeed_to_kph(maxspeed)
    if parsed > 0:
        return parsed, "maxspeed"
    return DEFAULT_SPEED_KPH.get(highway, 25.0), "default_by_highway"


def normalize_oneway(value: Any) -> str:
    if value is None:
        return "no"

    s = str(value).strip().lower()
    if s in {"yes", "true", "1"}:
        return "yes"
    if s in {"-1", "reverse"}:
        return "reverse"
    return "no"


def flatten_coords(geometry: Dict[str, Any]) -> List[Tuple[float, float]]:
    gtype = geometry.get("type")
    coords = geometry.get("coordinates", [])

    result: List[Tuple[float, float]] = []

    if gtype == "Point":
        return [(coords[0], coords[1])]

    if gtype == "LineString":
        return [(pt[0], pt[1]) for pt in coords]

    if gtype == "Polygon":
        if coords:
            return [(pt[0], pt[1]) for pt in coords[0]]
        return []

    if gtype == "MultiPoint":
        return [(pt[0], pt[1]) for pt in coords]

    if gtype == "MultiLineString":
        for line in coords:
            result.extend((pt[0], pt[1]) for pt in line)
        return result

    if gtype == "MultiPolygon":
        for poly in coords:
            if poly:
                result.extend((pt[0], pt[1]) for pt in poly[0])
        return result

    return []


def centroid(points: List[Tuple[float, float]]) -> Tuple[float, float]:
    if not points:
        return 0.0, 0.0
    lon_sum = sum(p[0] for p in points)
    lat_sum = sum(p[1] for p in points)
    n = len(points)
    return lon_sum / n, lat_sum / n


def geometry_to_lines(geometry: Dict[str, Any]) -> List[List[Tuple[float, float]]]:
    gtype = geometry.get("type")
    coords = geometry.get("coordinates", [])

    lines: List[List[Tuple[float, float]]] = []

    if gtype == "LineString":
        lines.append([(pt[0], pt[1]) for pt in coords])
    elif gtype == "MultiLineString":
        for line in coords:
            lines.append([(pt[0], pt[1]) for pt in line])

    return lines


# =========================
# Export nodes and edges
# =========================
def export_nodes_and_edges(roads_geojson_path: str) -> None:
    data = load_geojson(roads_geojson_path)
    features = data.get("features", [])

    node_id_map: Dict[str, int] = {}
    nodes_rows: List[Dict[str, Any]] = []
    edges_rows: List[Dict[str, Any]] = []

    next_node_id = 1
    next_edge_id = 1

    for feature in features:
        props = feature.get("properties", {}) or {}
        geometry = feature.get("geometry", {}) or {}

        highway = str(safe_get(props, "highway", "")).strip()
        if not highway:
            continue

        if highway in NOISY_HIGHWAYS:
            continue

        if highway not in ROADLIKE_HIGHWAYS:
            continue

        lines = geometry_to_lines(geometry)
        if not lines:
            continue

        osm_way_id = safe_get(props, "@id", "")
        if not osm_way_id:
            osm_way_id = safe_get(props, "id", "")

        name = safe_get(props, "name", "")
        ref = safe_get(props, "ref", "")
        maxspeed = safe_get(props, "maxspeed", "")
        oneway = normalize_oneway(safe_get(props, "oneway", "no"))
        lanes = safe_get(props, "lanes", "")
        access = safe_get(props, "access", "")
        bridge = safe_get(props, "bridge", "")
        tunnel = safe_get(props, "tunnel", "")
        surface = safe_get(props, "surface", "")
        junction = safe_get(props, "junction", "")
        service = safe_get(props, "service", "")

        resolved_speed_kph, speed_source = resolve_speed_kph(highway, maxspeed)
        all_tags_json = json.dumps(props, ensure_ascii=False, sort_keys=True)

        for line in lines:
            if len(line) < 2:
                continue

            line_node_ids: List[int] = []

            for lon, lat in line:
                key = coord_key(lon, lat)
                if key not in node_id_map:
                    node_id_map[key] = next_node_id
                    nodes_rows.append(
                        {
                            "node_id": next_node_id,
                            "lat": round(lat, 7),
                            "lon": round(lon, 7),
                        }
                    )
                    next_node_id += 1
                line_node_ids.append(node_id_map[key])

            for i in range(len(line) - 1):
                from_lon, from_lat = line[i]
                to_lon, to_lat = line[i + 1]

                u = line_node_ids[i]
                v = line_node_ids[i + 1]

                distance_m = haversine_m(from_lat, from_lon, to_lat, to_lon)

                base_row = {
                    "edge_id": next_edge_id,
                    "osm_way_id": osm_way_id,
                    "u": u,
                    "v": v,
                    "name": name,
                    "ref": ref,
                    "highway": highway,
                    "length_m": round(distance_m, 3),
                    "maxspeed_raw": maxspeed,
                    "resolved_speed_kph": round(resolved_speed_kph, 3),
                    "speed_source": speed_source,
                    "oneway": oneway,
                    "lanes": lanes,
                    "access": access,
                    "bridge": bridge,
                    "tunnel": tunnel,
                    "surface": surface,
                    "junction": junction,
                    "service": service,
                    "all_tags_json": all_tags_json,
                }

                if oneway == "reverse":
                    reverse_only = dict(base_row)
                    reverse_only["u"] = v
                    reverse_only["v"] = u
                    edges_rows.append(reverse_only)
                    next_edge_id += 1
                    continue

                edges_rows.append(base_row)
                next_edge_id += 1

                if oneway == "no":
                    reverse_row = dict(base_row)
                    reverse_row["edge_id"] = next_edge_id
                    reverse_row["u"] = v
                    reverse_row["v"] = u
                    edges_rows.append(reverse_row)
                    next_edge_id += 1

    with open(NODES_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["node_id", "lat", "lon"],
        )
        writer.writeheader()
        writer.writerows(nodes_rows)

    edge_fields = [
        "edge_id",
        "osm_way_id",
        "u",
        "v",
        "name",
        "ref",
        "highway",
        "length_m",
        "maxspeed_raw",
        "resolved_speed_kph",
        "speed_source",
        "oneway",
        "lanes",
        "access",
        "bridge",
        "tunnel",
        "surface",
        "junction",
        "service",
        "all_tags_json",
    ]

    with open(EDGES_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=edge_fields)
        writer.writeheader()
        writer.writerows(edges_rows)

    print(f"Exported {NODES_CSV}: {len(nodes_rows)} rows")
    print(f"Exported {EDGES_CSV}: {len(edges_rows)} rows")


# =========================
# Export POIs
# =========================
def export_pois(full_geojson_path: str) -> None:
    data = load_geojson(full_geojson_path)
    features = data.get("features", [])

    poi_rows: List[Dict[str, Any]] = []
    next_poi_id = 1

    for feature in features:
        props = feature.get("properties", {}) or {}
        geometry = feature.get("geometry", {}) or {}

        if not any(k in props for k in INTERESTING_POI_KEYS):
            continue

        points = flatten_coords(geometry)
        if not points:
            continue

        lon, lat = centroid(points)

        row = {
            "poi_id": next_poi_id,
            "osm_id": safe_get(props, "@id", safe_get(props, "id", "")),
            "name": safe_get(props, "name", ""),
            "lat": round(lat, 7),
            "lon": round(lon, 7),
            "amenity": safe_get(props, "amenity", ""),
            "tourism": safe_get(props, "tourism", ""),
            "railway": safe_get(props, "railway", ""),
            "public_transport": safe_get(props, "public_transport", ""),
            "aeroway": safe_get(props, "aeroway", ""),
            "shop": safe_get(props, "shop", ""),
            "leisure": safe_get(props, "leisure", ""),
            "office": safe_get(props, "office", ""),
            "building": safe_get(props, "building", ""),
            "addr_street": safe_get(props, "addr:street", ""),
            "addr_housenumber": safe_get(props, "addr:housenumber", ""),
            "addr_city": safe_get(props, "addr:city", ""),
            "all_tags_json": json.dumps(props, ensure_ascii=False, sort_keys=True),
        }

        # 昨天同样的噪声过滤
        if not row["name"] and row["amenity"] in {"bench", "waste_basket"}:
            continue

        poi_rows.append(row)
        next_poi_id += 1

    poi_fields = [
        "poi_id",
        "osm_id",
        "name",
        "lat",
        "lon",
        "amenity",
        "tourism",
        "railway",
        "public_transport",
        "aeroway",
        "shop",
        "leisure",
        "office",
        "building",
        "addr_street",
        "addr_housenumber",
        "addr_city",
        "all_tags_json",
    ]

    with open(POIS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=poi_fields)
        writer.writeheader()
        writer.writerows(poi_rows)

    print(f"Exported {POIS_CSV}: {len(poi_rows)} rows")


# =========================
# Main
# =========================
def main() -> None:
    print("Step 1/2: Export nodes and edges ...")
    export_nodes_and_edges(ROADS_GEOJSON)

    print("Step 2/2: Export POIs ...")
    export_pois(FULL_GEOJSON)

    print("\nDone.")
    print(f"- {NODES_CSV}")
    print(f"- {EDGES_CSV}")
    print(f"- {POIS_CSV}")


if __name__ == "__main__":
    main()

Step 1/2: Export nodes and edges ...
Exported nodes.csv: 2605 rows
Exported edges.csv: 3993 rows
Step 2/2: Export POIs ...
Exported pois.csv: 2976 rows

Done.
- nodes.csv
- edges.csv
- pois.csv
